# Instructions

Copy the output of the last block and drop it in `_includes/map.html`

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from time import sleep

In [2]:
df = pd.read_csv("SNAP Member Info - Sheet1.csv")
locations = df[["City", "State"]].dropna().drop_duplicates()

# Split rows where City and State contain "/" into separate rows
split_rows = []
keep_mask = []
for idx, row in locations.iterrows():
    if "/" in str(row["City"]):
        cities = [c.strip() for c in row["City"].split("/")]
        states = [s.strip() for s in row["State"].split("/")]
        for city, state in zip(cities, states):
            split_rows.append({"City": city, "State": state})
        keep_mask.append(False)
    else:
        keep_mask.append(True)

locations = pd.concat(
    [locations[keep_mask], pd.DataFrame(split_rows)],
    ignore_index=True
).drop_duplicates()
locations

,City,State
0,Cambridge,MA
1,Ithaca,NY
2,New Haven,CT
3,Birmingham,Al
4,Baltimore,MD
5,Arlington,VA
6,Berkeley,CA
7,Seattle,WA
8,Miami,FL
9,St. Louis,MO


In [3]:
geolocator = Nominatim(user_agent="snap_member_map")

def get_coords(row):
    query = f"{row['City']}, {row['State']}, USA"
    location = geolocator.geocode(query)
    sleep(1)  # respect Nominatim rate limit
    if location:
        return pd.Series({"lat": location.latitude, "lon": location.longitude})
    return pd.Series({"lat": None, "lon": None})

locations[["lat", "lon"]] = locations.apply(get_coords, axis=1)
locations

,City,State,lat,lon
0,Cambridge,MA,42.365635,-71.104002
1,Ithaca,NY,42.437418,-76.548372
2,New Haven,CT,41.308214,-72.925052
3,Birmingham,Al,33.520682,-86.802433
4,Baltimore,MD,39.290882,-76.610759
5,Arlington,VA,38.876933,-77.089309
6,Berkeley,CA,37.870839,-122.272863
7,Seattle,WA,47.603832,-122.330062
8,Miami,FL,25.774157,-80.193597
9,St. Louis,MO,38.625406,-90.190009


In [4]:
for _, row in locations.iterrows():
    print(f"[[{row['lat']:.6f}, {row['lon']:.6f}], '{row['City']}, {row['State']}'],")

[[42.365635, -71.104002], 'Cambridge, MA'],
[[42.437418, -76.548372], 'Ithaca, NY'],
[[41.308214, -72.925052], 'New Haven, CT'],
[[33.520682, -86.802433], 'Birmingham, Al'],
[[39.290882, -76.610759], 'Baltimore, MD'],
[[38.876933, -77.089309], 'Arlington, VA'],
[[37.870839, -122.272863], 'Berkeley, CA'],
[[47.603832, -122.330062], 'Seattle, WA'],
[[25.774157, -80.193597], 'Miami, FL'],
[[38.625406, -90.190009], 'St. Louis, MO'],
[[34.053691, -118.242766], 'Los Angeles, CA'],
[[41.661256, -91.529911], 'Iowa City, IA'],
[[39.952724, -75.163526], 'Philadelphia, PA'],
[[43.074690, -89.384166], 'Madison, WI'],
[[39.739236, -104.984862], 'Denver, CO'],
[[37.444329, -122.159847], 'Palo Alto, CA'],
[[40.712728, -74.006015], 'New York, NY'],
[[41.875562, -87.624421], 'Chicago, IL'],
[[42.358834, -71.057830], 'Boston, MA'],
[[29.758938, -95.367697], 'Houston, TX'],
[[42.376238, -71.235564], 'Waltham, MA'],
[[35.913154, -79.055780], 'Chapel Hill, NC'],
[[35.996653, -78.901805], 'Durham, NC'],
[[4